<div style="text-align: center;">
    <h1>Project 2: Python</h1>
    <p><strong>Instructor:</strong> David Osterrieder</p>
    <p><strong>Members:</strong> Bui Ngoc Quach, Yen-Fang Tseng, Benjamin Phumin Schmidhalter & Nikolas Arent</p>
    <p><strong>Due Date:</strong> 12/01/2025</p>
</div>

# Python Project Task 1

## 1. Install and Import required libraries

### 1.1 Install required libraries
Listed here for completeness. But often not necessary!

In [ ]:
# Install dependencies with pip install -r requirements.txt before starting Jupyter.

### 1.2 Import required libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
from collections import Counter
import yfinance as yf
import re

## 2. Inspect the data (Business Understanding & Data Understanding)

### 2.1 Load the .parquet and .csv files

In [ ]:
from pathlib import Path
DATA_DIR = Path("data")
lsedata = pd.read_parquet(DATA_DIR / "LSEDATA_2022.parquet")
instruments = pd.read_csv(DATA_DIR / "Instruments_LSEDATA_2022.csv")

### 2.2 Inspect the data

#### 2.2.1 Inspect lsedata

In [ ]:
print(lsedata.head())

The Date column represents the date of the trade. The ISIN column contains the unique identifier for each security. The Name column shows the name of the asset being traded. The Time column indicates the exact time of the trade. The TradeType column specifies whether the transaction was a "Buy" or "Sell." The Volume column represents the number of shares or units traded. The Price column shows the price at which the security was traded.

In [ ]:
print(lsedata.isnull().sum())

No empty values -> good!

In [ ]:
print(lsedata.dtypes)

Date should be changed from Object to Datetype and from Object to Datetype(time) for Time. Considering changing TradeType from Object to Category to save memoryspace.

#### 2.2.2 inspect instruments

In [ ]:
print(instruments.head())

Unnamed: 0, just an index. The ISIN column represents the unique identifier for each asset, while the Name column provides the full name of the asset. The Shortcode column contains an abbreviation or shorthand for the asset, and the Type column categorizes the asset as either "Equity" or "Fund.

In [ ]:
print(instruments.isnull().sum())

Shortcode has empty values for us no issue, because we don't need it for the analysis

In [ ]:
print(instruments.dtypes)

Data types good! Possible to convert Type to category-datatyp

#### 2.2.3 Merging

We need both tables for further analysis, so we will merge them. From table "lsedata" we need everything except the name. From table "instruments" we need ISIN and Type. The tables can be merged via ISIN, as this is a unique identifier.

## 3. Data preperation

### 3.1 Merge the data using the common key'ISIN'

In [ ]:
# we use an inner join because we need complete data, i.e. ISIN is present in both tables and therefore also other data
merged_data = lsedata.merge(instruments, on = 'ISIN', how = 'inner')

In [ ]:
# to reduce the amount of RAM used, we delete the two tables read in as they are now located together on merged_data
# For this reason, also new tables will not be created in the following, e.g. for changes to the data or the model. Even if this means that it is no longer possible to access the original data
del lsedata
del instruments

### 3.2 Calculate other necessary columns

#### 3.2.1 EuroVolume
ETF, Fund & Equity: is calculated by multiplying volume and price <br>
Bond: is calculated by multiplying volume (notional) and price (percentage price increase) divided by 100<br><br>
Note, as no currency details were given, it is assumed that the price or the notional was already given in euros.

In [ ]:
# For each row, the lambda-function checks whether it is a bond type, then divides by 100, otherwise by 1 (i.e. no effect)
merged_data['EuroVolume'] = merged_data['Volume'] * merged_data['Price'] / merged_data['Type'].apply(lambda x: 100 if x == 'Bond' else 1)

#### 3.2.2 Within/outside trading hours

In [ ]:
# convert Time to HH:MM:SS
merged_data['Time'] = pd.to_datetime(merged_data['Time'], format='%H:%M:%S', errors='coerce').dt.time

# define start and end for within time
start_time = pd.to_datetime('09:00:00').time()
end_time = pd.to_datetime('17:30:00').time()

# If the time of the trade is between start and end 1 otherwise 0
merged_data['within_time'] = np.where(
    (merged_data['Time'] >= start_time) &
    (merged_data['Time'] <= end_time), 1, 0
)

#### 3.2.3 Trade direction (Buy or Sell)

In [ ]:
print(merged_data['TradeType'].unique())

There is only buy and sell. We work with 0 and 1

In [ ]:
# if buy than 1 otherwise (sale) is 0
merged_data['IsBuy'] = np.where(merged_data['TradeType'] == 'Buy', 1, 0)

#### 3.2.4 Add Weekday

In [ ]:
# convert date to datetime
merged_data['Date'] = pd.to_datetime(merged_data['Date'], errors='coerce')
# determine the day and save on Weekday
merged_data['Weekday'] = merged_data['Date'].dt.weekday

### 3.2 Remove not necessary data (rows and columns)

#### 3.2.1 delete unnecessary columns

In [ ]:
merged_data = merged_data.drop(columns=['Name_x', 'Name_y', 'Unnamed: 0', 'Shortcode', 'Date', 'TradeType', 'Time'])

#### 3.2.2 Remove 'Subscription Right' trades
these should not be taken into account according to the task definition

In [ ]:
# deletes rows where Type == 'Subscription Right'
merged_data = merged_data.drop(merged_data[merged_data['Type'] == 'Subscription Right'].index)

#### 3.2.3 Remove rare, but large-in-size savings plan executions

The task states that the types 'Equity','ETF' and 'Fund' have outliers. We therefore look at the EuroVolume of these types.

In [ ]:
# Box Plot for EuroVolume
plt.figure(figsize=(10, 6))
# x='Type' defines the x-axis, y='EuroVolume' the y-axis. Then merged data is used if type is 'Equity', 'ETF' or 'Fund'
sns.boxplot(x='Type', y='EuroVolume', data=merged_data[merged_data['Type'].isin(['Equity', 'ETF', 'Fund'])])
# we only display data up to the 99 % quantile, as otherwise very large outliers ensure that nothing can be recognized
plt.ylim(0, merged_data['EuroVolume'].quantile(0.99))

# show the table
plt.title('EuroVolume Distribution by Type')
plt.show()

The chart shows that equity and ETFs have a smaller average euro volume than funds. In addition, there are many data points outside the upper whisker, which can be considered as outliers. <br>
<br>
We use the standard IQR method to remove the outlier.

In [ ]:
# Calculate IQR for each type
def remove_outliers(df, column):
    Q1 = df[column].quantile(0.25)
    Q3 = df[column].quantile(0.75)
    IQR = Q3 - Q1
    upper_bound = Q3 + 1.5 * IQR
    return df[df[column] <= upper_bound]

# Apply the function to each type except 'Bonds'
merged_data = merged_data.groupby('Type', group_keys=False).apply(lambda x: remove_outliers(x, 'EuroVolume') if x.name in ['Equity', 'ETF', 'Fund'] else x)

In [ ]:
# Verify the result
plt.figure(figsize=(10, 6))
sns.boxplot(x='Type', y='EuroVolume', data=merged_data[merged_data['Type'].isin(['Equity', 'ETF', 'Fund'])])

# Adjust for visibility
plt.title('EuroVolume Distribution by Type (After Removing Outliers)')
plt.show()

The graph shows that the data is now more evenly distributed. The data outside the upper whiskers are of course no longer outliers.

## 4. Build the linear regression model (Modeling)

### 4.1 Data Preprocessing: One-Hot Encoding and Feature Preparation for Regression

In [ ]:
# Convert categorical variables ('Weekday' and 'Type') to dummy variables
merged_data = pd.get_dummies(merged_data, columns=['Weekday', 'Type'], drop_first=True)

# One-hot encoding creates new columns for each category, dropping the first category
# The first category is used as a reference for the other categories

### 4.2 Feature selection and target definition for OLS Model

In [ ]:
# Define features (X) and target (y)
# The variables Weekday and Type are encoded with dummy variables for that we need all columns that have 'Weekday_' or 'Type_' in their name,
X = merged_data[['within_time', 'IsBuy'] + [col for col in merged_data.columns if 'Weekday_' in col or 'Type_' in col]]
y = merged_data['EuroVolume']

### 4.3 Ensuring Numeric Input: Converting Boolean and Categorical Variables

In [ ]:
# Ensure all columns in X are numeric
X = X.apply(pd.to_numeric, errors='coerce')

# Convert boolean columns to integers
X = X.astype({col: 'int32' for col in X.select_dtypes(include='bool').columns})

### 4.4 Statistical Model Fitting: OLS Regression on Market Data

In [ ]:
# Add a constant column for the intercept term in the OLS regression
X = sm.add_constant(X)

# Fit the OLS Model
model = sm.OLS(y, X)
results = model.fit()

# Display the Summary
print(results.summary())

R-squared value of 0.052, indicating that approximately 5.2% of the variation in EuroVolume is explained by the model. The F-statistic is highly significant (p-value < 0.001), demonstrating the overall statistical significance of the model. <br>
<br>
All independent variables are statistically significant.
<br>
The constant is 15.850, which means that if all variables are 0, the EuroVolume is 15.850.
<br>
within_time is approximately 27, i.e. if trading takes place between 9:00 and 17:30 o'clock, the trading volume increases by 27.
<br>
IsBuy is approximately -103, i.e. if the trade is a buy (and not a sale) the EuroVolume decreases by 103.
<br>
The weekdays Tuesday to Friday are between 17 and 34, which means that the trading volume increases on each day compared to Monday
<br>
The type of security (ETF, share, fund) is between -13,210 and -15,390, which means that the trading volume drops significantly compared to bonds

## 5. Evaluation
The model and all variables are statistically significant, demonstrating that all factors analyzed have an impact on euro trading volume. However, only 5.2 % is explained by the variables, which is a low value and therefore raises the question of which factors are more relevant in explaining the euro trading volume.<br><br>
It is also noticeable that the type has the greatest influence on the volume. It is therefore decisive for the trading volume whether bonds, ETFs, funds or equities are traded.<br><br>
The days of the week and the trading time have a similarly large influence. While the trading direction has a greater influence. The trading volume is smaller for buys than for sells.


# Task 2: Simplified retirement age planning
<strong>Short discribtion, how the code is progressed:</strong>

*   input variables are entered (with validation)
*   the function ‘return_calculation’ is called to calculate the return and volatility of ETF
* the function ‘montecarlo_lifespan is called
  * loop through the specified number of simulations
  * it estimates per round how old the person will be
  * then function working_years is called
    * there it is checked how many years the person has to work so that the money is enough
  * result is stored in function ‘montecarlo_lifespan’ in array, this is returned after loop
* how often each number of working years occurs is counted back, probability is calculated
* these are cumulated and output
* Graph is created that shows the years to work for different uncertainties

## 1. save death rates
Source: https://www.ssa.gov/oact/STATS/table4c6.html

In [ ]:
death_rates_male = [
    0.005860, 0.000420, 0.000272, 0.000225, 0.000184, 0.000157, 0.000140, 0.000128,
    0.000122, 0.000123, 0.000129, 0.000138, 0.000164, 0.000220, 0.000310, 0.000446,
    0.000637, 0.000868, 0.001100, 0.001270, 0.001373, 0.001488, 0.001605, 0.001714,
    0.001835, 0.001963, 0.002082, 0.002202, 0.002330, 0.002457, 0.002574, 0.002683,
    0.002787, 0.002881, 0.002974, 0.003074, 0.003175, 0.003295, 0.003444, 0.003608,
    0.003780, 0.003958, 0.004144, 0.004337, 0.004540, 0.004774, 0.005064, 0.005399,
    0.005796, 0.006214, 0.006671, 0.007167, 0.007736, 0.008351, 0.009035, 0.009770,
    0.010567, 0.011398, 0.012291, 0.013224, 0.014267, 0.015353, 0.016484, 0.017617,
    0.018759, 0.019914, 0.021104, 0.022423, 0.023847, 0.025357, 0.027050, 0.028970,
    0.031188, 0.033754, 0.036747, 0.040563, 0.044308, 0.048498, 0.053229, 0.058778,
    0.064617, 0.070947, 0.077834, 0.085686, 0.094809, 0.105090, 0.116592, 0.129306,
    0.142732, 0.157638, 0.174458, 0.193027, 0.212930, 0.232657, 0.251826, 0.270943,
    0.289756, 0.307998, 0.325393, 0.341662, 0.358746, 0.376683, 0.395517, 0.415293,
    0.436058, 0.457860, 0.480753, 0.504791, 0.530031, 0.556532, 0.584359, 0.613577,
    0.644256, 0.676468, 0.710292, 0.745806, 0.783097, 0.822251, 0.863364, 0.906532,
    1
]

death_rates_female = [
    0.005063, 0.000393, 0.000223, 0.000177, 0.000144, 0.000122, 0.000109, 0.000102,
    0.000098, 0.000097, 0.000103, 0.000113, 0.000131, 0.000157, 0.000190, 0.000233,
    0.000291, 0.000355, 0.000418, 0.000461, 0.000507, 0.000556, 0.000610, 0.000666,
    0.000722, 0.000775, 0.000831, 0.000889, 0.000952, 0.001025, 0.001104, 0.001192,
    0.001289, 0.001383, 0.001465, 0.001544, 0.001626, 0.001719, 0.001824, 0.001940,
    0.002066, 0.002202, 0.002351, 0.002482, 0.002622, 0.002789, 0.002994, 0.003219,
    0.003467, 0.003729, 0.004011, 0.004306, 0.004634, 0.004981, 0.005370, 0.005831,
    0.006326, 0.006837, 0.007399, 0.008033, 0.008687, 0.009411, 0.010139, 0.010849,
    0.011550, 0.012216, 0.012952, 0.013844, 0.014863, 0.016028, 0.017329, 0.018859,
    0.020609, 0.022620, 0.024958, 0.027906, 0.030925, 0.034140, 0.037620, 0.041725,
    0.046324, 0.051334, 0.056911, 0.063279, 0.070704, 0.079184, 0.088697, 0.099240,
    0.110480, 0.123078, 0.137152, 0.152605, 0.169494, 0.187623, 0.206647, 0.225890,
    0.245054, 0.263815, 0.281828, 0.298738, 0.316662, 0.335662, 0.355802, 0.377150,
    0.399779, 0.423766, 0.449192, 0.476143, 0.504712, 0.534994, 0.567094, 0.601120,
    0.637187, 0.675418, 0.710292, 0.745806, 0.783097, 0.822251, 0.863364, 0.906532,
    1
]

## 2. monte carlo simulation

In [ ]:
def montecarlo_lifespan():
    num_simulations = answers["num_simulations"]

    # empty array to safe working year list
    working_year_list = []

    # find out which death rates are required
    if answers["gender"] == "male":
        death_rates = death_rates_male
    else:
        death_rates = death_rates_female

    # loop throug number of simulations
    for _ in range(num_simulations):
        current_age_sim = int(answers["initial_age"])

        # simulate every year until person dies
        while current_age_sim < len(death_rates):

            # random number between 0 and 1
            random_number = np.random.uniform(0, 1)

            # probability that person dies at age (current_age_sim)
            p_dies = death_rates[current_age_sim]

            # break while-loop if person dies
            if random_number <= p_dies:
                break

            # persons lifes one year longer
            current_age_sim += 1

        # append list how long person has to work (calculation with function "working_years")
        work_years = working_years(current_age_sim)
        working_year_list.append(work_years)

    return working_year_list

## 3. Calculate how many years person has to work

In [ ]:
def working_years(life_expectancy):
    # save input parameters to variables
    initial_age = answers["initial_age"]
    initial_wealth = answers["initial_wealth"]
    annual_contribution = answers["annual_contribution"]
    annual_withdrawal = answers["annual_withdrawal"]
    inflation_rate = answers["inflation_rate"]
    exp_return = answers["exp_return"]
    volatility = answers['volatility']

    wealth = initial_wealth

    # If the person dies in the same year that is specified as the initial age, then 0 is returned.
    # 0 as the person no longer has to work.
    if initial_age == life_expectancy:
      return 0

    # If the person were to stop working now, how much money would the person need?
    # Sum of money required annually (taking into account the inflation rate) for expected remaining years of life
    required_wealth = sum(annual_withdrawal*((1+inflation_rate) ** r)  for r in range(1, life_expectancy-initial_age))

    # look if persons has already enough money without working one more year
    if required_wealth < wealth:
        return 0

    # loop through years of life the person has
    for x in range(initial_age+1, life_expectancy):

        # Simulate the annual return based on a normal distribution
        random_return = np.random.normal(exp_return, volatility)

        # update wealth (first annual return on assets and then the payment per year)
        wealth *= (1 + random_return)
        wealth += annual_contribution
        # update required wealth (the money of the year that was still worked is deducted)
        required_wealth -= annual_withdrawal*((1+inflation_rate) ** (x-initial_age))

        # If the required wealth is smaller than the wealth of the person, then the person can retire (result necessary working years)
        if  required_wealth < wealth:
            return (x-initial_age)

## 4. calculate the average return and volatilty

In [ ]:
def return_calculation():
    # define ticker-symbol and time-frame
    ticker = "SPY"
    start_date = answers["start_date"]
    end_date = answers["end_date"]

    # download data from yahoo-finance
    data = yf.download(ticker, start=start_date, end=end_date, auto_adjust=False, multi_level_index=False)
    print(data.columns)
    # calculate daily return
    data['Daily Return'] = data['Adj Close'].pct_change()

    # calculate mean of daily return
    average_daily_return = data['Daily Return'].mean()

    # calculate the yearly-return (252 trading days)
    annual_return = average_daily_return * 252

    # calculate the daily volalitiy
    daily_volatility = data['Daily Return'].std()

    # calculate the yearly volality (252 trading days)
    # np.sqrt, because volatility is scaled quadratically
    annual_volatility = daily_volatility * np.sqrt(252)

    return annual_return, annual_volatility

## 5. input

### 5.1 questions

In [ ]:
questions = [
    ("How many simulations should the program run to calculate the result?", 50000, "num_simulations"),
    ("What is the starting age?", 55, "initial_age"),
    ("What is the starting wealth?", 200000, "initial_wealth"),
    ("How much money will be invested at the end of each year (if decided to keep working)?", 24000, "annual_contribution"),
    ("How much money does the person need each year once he is retired?", 30000, "annual_withdrawal"),
    ("Which constant inflation rate is assumed to adjust the withdrawal rate?", 0.025, "inflation_rate"),
    ("How much uncertainty is the person willing to accept regarding the possibility of running out of money?", 0.05, "uncertainty"),
    ("What is the gender", "male", "gender"),
    ("What should be the start date to calculate the return for the ETF?", "1960-01-01" ,"start_date"),
    ("What should be the end date to calculate the return for the ETF?", "2023-12-31" ,"end_date")
]

### 5.2 function to check if input is number

In [ ]:
def is_number(value):
    try:
        if int(value) >= 0:
            return True
        else:
          return False
    except ValueError:
        return False

def is_valid_date_format(date_str):
    # regex: 4 numbers - 2 numbers - 2 numbers
    pattern = r"^\d{4}-\d{2}-\d{2}$"
    return bool(re.match(pattern, date_str))

### 5.3 record input and show results

In [ ]:
answers = {}
print("\033[1mInstruction:\033[0m You can enter the desired input parameters below. If you want to use the input of task two (i.e. default), simply click enter.")
print("")

# enter input
# loop through the defined questions
for question, default, key in questions:
    valid = False
    # Loop is only cancelled when valid input is given
    while not valid:
        # display question (input)
        answer = input(f"{question} (Default: {default}) ")

        # if user don't enter anything default value is used
        if answer == "":
            answers[key] = default
            valid = True

        # if the question is gender, it is checked whether it is a valid gender
        elif key == "gender":
            if answer.strip().lower() in ["male", "female"]:
              answers[key] = answer.strip().lower()
              valid = True
            else:
              print("\033[31mNo valid input! Please enter 'male' or 'female'.\033[0m")

        # if the question is age, it is checked whether the age is between 0 and 120
        elif key == "initial_age":
            if 0 <= int(answer) <= 120:
                answers[key] = int(answer)
                valid = True
            else:
              print("\033[31mNo valid input! Please enter a number between  0 and 120.\033[0m")

        # if the question is inflation rate or uncertainty, then check whether the number is between 0 and 1
        elif key in ["inflation_rate", "uncertainty"]:
            if 0 <= float(answer.replace(',', '.')) <= 1:
                answers[key] = float(answer.replace(',', '.'))
                valid = True
            else:
              print("\033[31mNo valid input! Please enter a number between 0 and 1 (format: 0.00).\033[0m")

        # if the question ist start-date check if it is the right format YYYY-MM-DD
        # no need to check whether there is still data for the period (i.e. if it is too far back), the last available data is then simply used
        elif key == "start_date":
            if is_valid_date_format(answer):
                answers[key] = answer
                valid = True
            else:
              print("\033[31mNo valid input! Please enter a date in the format YYYY-MM-DD.\033[0m")

        # if the question ist end-date check if it is the right format YYYY-MM-DD & that end date is after start date
        elif key == "end_date":
            if is_valid_date_format(answer) & (answer > answers["start_date"]):
                answers[key] = answer
                valid = True
            else:
              print("\033[31mNo valid input! Please enter a date in the format YYYY-MM-DD and after the start date.\033[0m")

        # all remaining entries are numbers, therefore test whether they are numbers above 0
        elif is_number(answer):  # check if number
            answers[key] = int(answer)
            valid = True

        # error message if user input contains e.g. letters
        else:
            print("\033[31mNo valid input! Please enter a number (>= 0) or press Enter for default.\033[0m")

# calculate return and volatilty with return_calculation function
exp_return, volatility = return_calculation()
# safe the results in 'answers'
answers['exp_return'] = exp_return
answers['volatility'] = volatility

# Monte Carlo simulation of lifetime
work_years = montecarlo_lifespan()

# Analyze results
frequencies = Counter(work_years)  # Count occurrences of each year
probabilities = {
    years: frequency / answers["num_simulations"]
    for years, frequency in frequencies.items()
    if years is not None
}

# Sort the working years
sorted_working_years = sorted(probabilities.items())

# variable initialization
cumulative_probability = 0
required_years = None
uncertainty = answers["uncertainty"]
years_with_certainty = []

# Find years with cumulative probability between 75 % and 100 %
for year, probability in sorted_working_years:
    # calculate cummulative probability
    cumulative_probability += probability
    # if the cumulative probability is between 75 % and 100 %, the year and the probability are saved in a list
    # for later graph output
    if 0.75 <= cumulative_probability <= 1:
        years_with_certainty.append((year, cumulative_probability))
    # save how many years he has to work, for specified uncertainty in input
    # check with "required_years is None", that the result is not already stored otherwise the year would not count, but become the maximum number
    if (cumulative_probability >= (1 - answers["uncertainty"])) & (required_years is None):
        required_years = year

# output
print("")
print("\033[1mResults:\033[0m")
print(f"\nIf you accept {uncertainty:.2%} probability of running out of money in your lifetime, then you should work for {required_years} years.")
print("")

# Extract data for the graph
years = [item[0] for item in years_with_certainty]
certainty = [1 - item[1] for item in years_with_certainty]

# Create graph
plt.figure(figsize=(8, 6))
plt.step(certainty, years, where='post', linestyle='-', color='black')
plt.xlabel('Accepted Uncertainty')
plt.ylabel('Working Years')
plt.title('years of work in relation to the risk of running out of money')
plt.show()